In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors2.db'
)

query = 'I just discovered the course. Can I still join it?'
query_vector = model.encode(query)

results = vs_index.search(
    query_vector,
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

from rag_helper import RAGBase

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {'course': self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )
    

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
)

result = vector_assistant.rag('the program has already begun, can I still sign up?')
print(result)

result = vector_assistant.rag('whats queen gambit?')
print(result)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Yes — you can still sign up and start learning. If you want a certificate, make sure to submit your project while submissions are still open.
I don't know.
